# 05 Plotting: Causal Interpretations

This notebook aggregates encoding and decoding feature-importance results and visualizes their overlap for causal interpretation across BR and Replay conditions.

Included function:
- `causal_overview(df_joint, subject, condition)`: builds a region-level interpretation table for one subject and condition, combining encoding/decoding relevance and p-values.

Main analysis steps in this notebook:
1. Load  (`df_joint`) for ICA or atlas-based features.
2. Aggregate significant regions across subjects.
3. Create glass-brain and slice-based visualizations.
4. Extract region labels and produce subject-level causal interpretation summaries.

In [ ]:
import pandas as pd
import numpy as np

df_encoding = pd.read_csv('/your/path/joint_enc_dec_atlas.csv') # update this path to the atlas features
# df_decoding = pd.read_csv('your/path/joint_enc_dec_ica.csv') # update this path to the ICA features

In [ ]:
# Create joint dataframe combining encoding and decoding results
df_joint = pd.merge(
    df_decoding[['subject', 'cycle_pair', 'mean_accuracy', 'num_significant_features', 'significant_features', 'p_values']],
    df_encoding[['subject', 'cycle_pair', 'n_samples', 'num_significant_features', 'significant_features', 'p_values']],
    on=['subject', 'cycle_pair'],
    how='inner',
    suffixes=('_dec', '_enc')
)

# Rename columns to match desired format
df_joint = df_joint.rename(columns={
    'num_significant_features_enc': 'n_enc_features',
    'significant_features_enc': 'enc_feature_idx',
    'p_values_enc': 'enc_p_values',
    'num_significant_features_dec': 'n_dec_features',
    'significant_features_dec': 'dec_feature_idx',
    'p_values_dec': 'dec_p_values'
})

# Reorder columns
df_joint = df_joint[['subject', 'cycle_pair', 'n_samples', 'mean_accuracy', 
                     'n_enc_features', 'enc_feature_idx', 'enc_p_values',
                     'n_dec_features', 'dec_feature_idx', 'dec_p_values']]

# Clean p_values by removing np.float64 wrappers
df_joint['enc_p_values'] = df_joint['enc_p_values'].str.replace(r'np\.float64\(', '', regex=True).str.replace(r'\)', '', regex=True)
df_joint['dec_p_values'] = df_joint['dec_p_values'].str.replace(r'np\.float64\(', '', regex=True).str.replace(r'\)', '', regex=True)

df_joint

In [ ]:
from nilearn.datasets import fetch_atlas_schaefer_2018
all_subjects = df_joint['subject'].unique().tolist()
features = fetch_atlas_schaefer_2018(n_rois=400, resolution_mm=2, verbose=1)

In [ ]:
# aggregate significant regions and create brain visualizations
from nilearn import plotting
import matplotlib.pyplot as plt
import nibabel as nib
import ast

# load atlas image
atlas_img = nib.load(features.maps)
atlas_data = atlas_img.get_fdata()
n_regions = int(atlas_data.max())  # regions are labeled 1 to n_regions
n_subjects = len(all_subjects)

print(f"Atlas has {n_regions} regions")
print(f"Number of subjects: {n_subjects}")

# initialize count arrays for encoding and decoding (BR and Replay)
enc_br_counts = np.zeros(n_regions, dtype=int)
enc_replay_counts = np.zeros(n_regions, dtype=int)
dec_br_counts = np.zeros(n_regions, dtype=int)
dec_replay_counts = np.zeros(n_regions, dtype=int)

# count occurrences of each region in selected indices
for _, row in df_joint.iterrows():
    # Parse feature indices (handle both list and string formats)
    enc_selected = row['enc_feature_idx']
    if isinstance(enc_selected, str):
        enc_selected = ast.literal_eval(enc_selected)
    
    dec_selected = row['dec_feature_idx']
    if isinstance(dec_selected, str):
        dec_selected = ast.literal_eval(dec_selected)
    
    # Count encoding features
    if row['cycle_pair'] == 'BR':
        enc_br_counts[enc_selected] += 1
    else:
        enc_replay_counts[enc_selected] += 1
    
    # Count decoding features
    if row['cycle_pair'] == 'BR':
        dec_br_counts[dec_selected] += 1
    else:
        dec_replay_counts[dec_selected] += 1

print(f"\nEncoding BR: {np.sum(enc_br_counts > 0)} regions significant in at least 1 subject (max={enc_br_counts.max()})")
print(f"Encoding Replay: {np.sum(enc_replay_counts > 0)} regions significant in at least 1 subject (max={enc_replay_counts.max()})")
print(f"Decoding BR: {np.sum(dec_br_counts > 0)} regions significant in at least 1 subject (max={dec_br_counts.max()})")
print(f"Decoding Replay: {np.sum(dec_replay_counts > 0)} regions significant in at least 1 subject (max={dec_replay_counts.max()})")

# create new data arrays with counts
enc_br_data = np.zeros_like(atlas_data)
enc_replay_data = np.zeros_like(atlas_data)
dec_br_data = np.zeros_like(atlas_data)
dec_replay_data = np.zeros_like(atlas_data)

# map counts to atlas regions (region labels are 1-indexed)
for region_idx in range(n_regions):
    region_label = region_idx + 1  # atlas labels start at 1
    mask = atlas_data == region_label
    enc_br_data[mask] = enc_br_counts[region_idx]
    enc_replay_data[mask] = enc_replay_counts[region_idx]
    dec_br_data[mask] = dec_br_counts[region_idx]
    dec_replay_data[mask] = dec_replay_counts[region_idx]

# create nifti images
enc_br_img = nib.Nifti1Image(enc_br_data, atlas_img.affine, atlas_img.header)
enc_replay_img = nib.Nifti1Image(enc_replay_data, atlas_img.affine, atlas_img.header)
dec_br_img = nib.Nifti1Image(dec_br_data, atlas_img.affine, atlas_img.header)
dec_replay_img = nib.Nifti1Image(dec_replay_data, atlas_img.affine, atlas_img.header)

# create legend for subject count
from matplotlib.patches import Patch
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap

# Create custom colormap from light red to bright yellow
colors = [
    (1.0, 1.0, 0.0),  # bright yellow
    (1.0, 0.7, 0.0),  # orange
    (1.0, 0.4, 0.0),  # red-orange
    (0.8, 0.2, 0.2),  # deep red
    (0.4, 0.0, 0.0)   # dark red
]
cmap_yellow_to_red = LinearSegmentedColormap.from_list("yellow_to_red", colors)
cmap_red_to_yellow = LinearSegmentedColormap.from_list("red_to_yellow", colors)

max_subjects = max(enc_br_counts.max(), enc_replay_counts.max(), dec_br_counts.max(), dec_replay_counts.max())
legend_handles = []
for i in range(1, max_subjects + 1):
    color_val = i / max_subjects  # normalize to [0, 1]
    color = cmap_red_to_yellow(color_val)
    legend_handles.append(Patch(color=color, label=f'{i} subject{"s" if i > 1 else ""}'))

# glass brain visualization with 4 plots (2x2 layout)
fig = plt.figure(figsize=(20, 20))
v_max = max_subjects
colorbar = False
# Encoding BR (top left)
ax1 = plt.subplot(4, 1, 1)
display_enc_br = plotting.plot_glass_brain(
    enc_br_img,
    display_mode='lyrz',
    colorbar=colorbar,
    cmap=cmap_red_to_yellow,
    vmax=v_max,
    vmin=0,
    plot_abs=False,
    threshold=0.5,
    axes=ax1
)

# Encoding Replay (top right)
ax2 = plt.subplot(4, 1, 2)
display_enc_replay = plotting.plot_glass_brain(
    enc_replay_img,
    display_mode='lyrz',
    colorbar=colorbar,
    cmap=cmap_red_to_yellow,
    vmax=v_max,
    vmin=0,
    plot_abs=False,
    threshold=0.5,
    axes=ax2
)

# Decoding BR (bottom left)
ax3 = plt.subplot(4, 1, 3)
display_dec_br = plotting.plot_glass_brain(
    dec_br_img,
    display_mode='lyrz',
    colorbar=colorbar,
    cmap=cmap_red_to_yellow,
    vmax=v_max,
    vmin=0,
    plot_abs=colorbar,
    threshold=0.5,
    axes=ax3
)

# Decoding Replay (bottom right)
ax4 = plt.subplot(4, 1, 4)
display_dec_replay = plotting.plot_glass_brain(
    dec_replay_img,
    display_mode='lyrz',
    colorbar=colorbar,
    cmap=cmap_red_to_yellow,
    vmax=v_max,
    vmin=0,
    plot_abs=False,
    threshold=0.5,
    axes=ax4
)

# Add legend to the figure
fig.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(0.92, 0.5), fontsize=20)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.show()

In [ ]:
# Axial slice visualization showing significant regions across all subjects
# Same data as glass brain plots but with multiple axial slices

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# Use the same colormap and data from previous cell
# (enc_br_img, enc_replay_img, dec_br_img, dec_replay_img already created)

# Create figure with 4 rows (one for each condition combination)
fig, axes = plt.subplots(4, 1, figsize=(20, 24))

# Define slice coordinates (12 evenly spaced axial slices)
cut_coords = 5

# Encoding BR
display_enc_br = plotting.plot_stat_map(
    enc_br_img,
    display_mode='z',
    cut_coords=cut_coords,
    colorbar=False,
    cmap=cmap_red_to_yellow,
    vmax=max_subjects,
    vmin=0,
    threshold=0.5,
    axes=axes[0]
)

# Encoding Replay
display_enc_replay = plotting.plot_stat_map(
    enc_replay_img,
    display_mode='z',
    cut_coords=cut_coords,
    colorbar=False,
    cmap=cmap_red_to_yellow,
    vmax=max_subjects,
    vmin=0,
    threshold=0.5,
    axes=axes[1]
)

# Decoding BR
display_dec_br = plotting.plot_stat_map(
    dec_br_img,
    display_mode='z',
    cut_coords=cut_coords,
    colorbar=False,
    cmap=cmap_red_to_yellow,
    vmax=max_subjects,
    vmin=0,
    threshold=0.5,
    axes=axes[2]
)

# Decoding Replay
display_dec_replay = plotting.plot_stat_map(
    dec_replay_img,
    display_mode='z',
    cut_coords=cut_coords,
    colorbar=False,
    cmap=cmap_red_to_yellow,
    vmax=max_subjects,
    vmin=0,
    threshold=0.5,
    axes=axes[3]
)

plt.tight_layout()
plt.show()

In [ ]:
# for subject 23 get the labels from the atlas for the corresponding significant regions 
data_s23 = df_joint[df_joint['subject'] == 's23'] 
enc_idxs = data_s23.enc_feature_idx
dec_idxs = data_s23.dec_feature_idx

In [ ]:
# get labels of significant features for encoding
import ast

# Collect all encoding indices for s23
all_enc_indices = set()
for idx in data_s23.enc_feature_idx:
    if isinstance(idx, str):
        idx = ast.literal_eval(idx)
    all_enc_indices.update(idx)

# Get the labels
encoding_labels = [features.labels[i] for i in sorted(all_enc_indices)]
print("Encoding significant regions for s23:")
for i, label in zip(sorted(all_enc_indices), encoding_labels):
    print(f"{i}: {label}")

In [ ]:
def causal_overview(df_joint, subject="s23", condition="Replay"):
    import ast
    import re
    
    df_sub = df_joint[(df_joint["subject"] == subject) &
                      (df_joint["cycle_pair"] == condition)].iloc[0]
    
    # Parse feature indices and p-values, handling NaN and numpy types
    def safe_parse_list(s):
        if pd.isna(s):
            return []
        # Replace 'nan' with 'None' for ast.literal_eval
        s = re.sub(r'\bnan\b', 'None', s)
        return ast.literal_eval(s)
    
    enc_features = safe_parse_list(df_sub["enc_feature_idx"])
    enc_p_values = safe_parse_list(df_sub["enc_p_values"])
    
    dec_features = safe_parse_list(df_sub["dec_feature_idx"])
    dec_p_values = safe_parse_list(df_sub["dec_p_values"])
    
    enc_idx = set(enc_features)
    dec_idx = set(dec_features)
    
    all_features = sorted(set(enc_idx.union(dec_idx)))
    
    rows = []
    
    for f in all_features:
        enc = f in enc_idx
        dec = f in dec_idx
        
        # Get p-values
        enc_p = enc_p_values[f] if enc else None
        dec_p = dec_p_values[f] if dec else None
        
        # Get region name
        region_name = features.labels[f]
        
        # ------------------------
        # Stimulus-based logic
        # ------------------------
        if condition == "Replay":
            
            # Encoding interpretation
            enc_interp = "Effect of S" if enc else "Not effect of S"
            # Decoding interpretation
            dec_interp = "Inconclusive"
            
            # Combined
            if enc and dec:
                combined = "Effect of S"
            elif enc and not dec:
                combined = "Indirect effect of S"
            elif not enc and dec:
                combined = "Brain state context"
            else:
                combined = "Irrelevant"
        
        # ------------------------
        # Response-based logic
        # ------------------------
        elif condition == "BR":
            
            # Encoding interpretation
            enc_interp = "Inconclusive" if enc else "Not cause of R"
            # Decoding interpretation
            dec_interp = "Inconclusive"
            
            # Combined
            if enc and dec:
                combined = "Inconclusive"
            elif enc and not dec:
                combined = "No direct cause of R"
            elif not enc and dec:
                combined = "Brain state context"
            else:
                combined = "Irrelevant"
        
        rows.append({
            'subject': subject,
            'cycle_pair': condition,
            "Feature": f,
            "Region Name": region_name,
            "Encoding Relevant": enc,
            "Decoding Relevant": dec,
            "Encoding P-value": enc_p,
            "Decoding P-value": dec_p,
            "Encoding Interpretation": enc_interp,
            "Decoding Interpretation": dec_interp,
            "Combined Interpretation": combined
        })
    
    return pd.DataFrame(rows).sort_values("Feature")


# -------------------------------
# Generate both tables
# -------------------------------
stimulus_based_table = causal_overview(df_joint, subject="s23", condition="Replay")
response_based_table = causal_overview(df_joint, subject="s23", condition="BR")

stimulus_based_table.to_csv('/gpfs01/bartels/user/smuehlinghaus/causalcoding/Results/Causal_Interpretation/stimulus_based_table_s23.csv')
response_based_table.to_csv('/gpfs01/bartels/user/smuehlinghaus/causalcoding/Results/Causal_Interpretation/responsed_based_table_s23.csv')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Consistent color mapping aligned with your final logic

color_map = {
    # -----------------------
    # Stimulus-based (Replay)
    # -----------------------
    "Effect of S": "#2ca02c",          # green
    "Indirect effect of S": "#ff7f0e", # orange
    "Brain state context": "#1f77b4",  # blue
    "Irrelevant": "#7f7f7f",           # gray
    
    # -----------------------
    # Response-based (BR)
    # -----------------------
    "No direct cause of R": "#ff7f0e", # orange (analogous to indirect)
    "Inconclusive": "#9467bd"          # purple
}

In [ ]:
atlas_17 = fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=17, resolution_mm=2)
atlas_17.lut

In [ ]:
def plot_causal_heatmap(df, subject="s23", condition="Replay"):
    
    df_sub = df[(df["subject"] == subject) &
                (df["cycle_pair"] == condition)].copy()
    
    # Extract network
    df_sub["Network"] = df_sub["Region Name"].str.split("_").str[2]
    
    # Count causal roles per network
    role_counts = (
        df_sub.groupby(["Network", "Combined Interpretation"])
              .size()
              .unstack(fill_value=0)
    )
    
    plt.figure(figsize=(10,6))
    sns.heatmap(role_counts, annot=True, cmap="viridis")
    
    title_type = "Stimulus-based (Replay)" if condition == "Replay" else "Response-based (BR)"
    
    plt.title(f"{subject} – {title_type}\nCausal Role Distribution per Network")
    plt.ylabel("Network")
    plt.xlabel("Causal Role")
    plt.tight_layout()
    plt.show()

def plot_feature_causal_map(df, subject="s23", condition="Replay"):
    
    df_sub = df[(df["subject"] == subject) &
                (df["cycle_pair"] == condition)].copy()
    
    df_sub["Network"] = df_sub["Region Name"].str.split("_").str[2]
    
    plt.figure(figsize=(12,8))
    
    for _, row in df_sub.iterrows():
        plt.scatter(
            row["Feature"],
            row["Network"],
            color=color_map.get(row["Combined Interpretation"], "black"),
            s=120
        )
    
    title_type = "Stimulus-based (Replay)" if condition == "Replay" else "Response-based (BR)"
    
    plt.title(f"{subject} – {title_type}\nFeature-Level Causal Roles")
    plt.xlabel("Feature Index")
    plt.ylabel("Network")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

def plot_side_by_side(df, subject="s23"):
    
    fig, axes = plt.subplots(1, 2, figsize=(16,8), sharey=True)
    
    for ax, condition in zip(axes, ["Replay", "BR"]):
        
        df_sub = df[(df["subject"] == subject) &
                    (df["cycle_pair"] == condition)].copy()
        
        df_sub["Network"] = df_sub["Region Name"].str.split("_").str[2]
        
        for _, row in df_sub.iterrows():
            ax.scatter(
                row["Feature"],
                row["Network"],
                color=color_map.get(row["Combined Interpretation"], "black"),
                s=100
            )
        
        title_type = "Stimulus-based (Replay)" if condition == "Replay" else "Response-based (BR)"
        ax.set_title(title_type)
        ax.set_xlabel("Feature Index")
        ax.grid(alpha=0.2)
    
    axes[0].set_ylabel("Network")
    
    plt.suptitle(f"{subject} – Causal Role Comparison")
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmaps
plot_causal_heatmap(stimulus_based_table, subject="s23", condition="Replay")
plot_causal_heatmap(response_based_table, subject="s23", condition="BR")

In [ ]:
s23 = df_joint[df_joint['subject'] == 's23']
s23

In [ ]:
import ast
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# subject-specific overlap maps (encoding vs decoding), split by BR and Replay
subject_ids = ['s08', 's23', 's27', 's28']

for subject_id in subject_ids:
    subject_rows = df_joint[df_joint['subject'] == subject_id]

    # discrete colormap: 1=blue, 2=orange, 3=green
    cmap = ListedColormap(['#1f77b4', '#1fa832', "#ce1818"])

    legend_handles = [
        Patch(color='#1f77b4', label='Encoding only'),
        Patch(color="#1fa832", label='Decoding only'),
        Patch(color="#ce1818", label='Encoding & Decoding')
    ]

    # Get atlas labels
    atlas_labels = features.labels

    for cycle_pair in ['BR', 'Replay']:
        cycle_rows = subject_rows[subject_rows['cycle_pair'] == cycle_pair]

        enc_indices = set()
        dec_indices = set()

        for _, row in cycle_rows.iterrows():
            enc = row['enc_feature_idx']
            if isinstance(enc, str):
                enc = ast.literal_eval(enc)
            enc_indices.update(enc)

            dec = row['dec_feature_idx']
            if isinstance(dec, str):
                dec = ast.literal_eval(dec)
            dec_indices.update(dec)

        # mutually exclusive categories
        only_enc = enc_indices - dec_indices
        only_dec = dec_indices - enc_indices
        both = enc_indices & dec_indices

        print(f"{subject_id} {cycle_pair} - Only encoding: {len(only_enc)} | Only decoding: {len(only_dec)} | Both: {len(both)}")

        # build region-coded volume: 1=only enc, 2=only dec, 3=both
        region_labels = np.zeros(n_regions, dtype=int)
        for idx in only_enc:
            region_labels[idx] = 1
        for idx in only_dec:
            region_labels[idx] = 2
        for idx in both:
            region_labels[idx] = 3

        subject_map = np.zeros_like(atlas_data)
        for region_idx in range(n_regions):
            region_label = region_idx + 1  # atlas labels start at 1
            mask = atlas_data == region_label
            subject_map[mask] = region_labels[region_idx]

        subject_img = nib.Nifti1Image(subject_map, atlas_img.affine, atlas_img.header)

        fig = plt.figure(figsize=(18, 10))
        fig.subplots_adjust(right=0.7, bottom=0.3)

        paradigm = 'stimulus-based' if cycle_pair == 'Replay' else 'response-based'

        display = plotting.plot(
            subject_img,
            display_mode='lyrz',
            colorbar=False,
            cmap=cmap,
            vmin=0.5,
            vmax=3.5,
            title=f'{subject_id} {cycle_pair} ({paradigm}): Encoding vs Decoding',
            threshold=0.5
        )

        plt.legend(handles=legend_handles, loc='center left', bbox_to_anchor=(1.02, 0.5), borderaxespad=0.0)
        
        # Create comprehensive legend table with region names
        all_indices = sorted(list(only_enc | only_dec | both))
        
        # Create table with columns
        legend_lines = []
        legend_lines.append(f"{'Index':<8} {'Region Name':<40} {'Category':<15}")
        legend_lines.append("="*65)
        
        for idx in all_indices:
            region_name = atlas_labels[idx] if idx < len(atlas_labels) else f"Region_{idx}"
            if idx in only_enc:
                cat = "Encoding"
            elif idx in only_dec:
                cat = "Decoding"
            else:
                cat = "Both"
            legend_lines.append(f"{idx:<8} {region_name:<40} {cat:<15}")
        
        legend_text = '\n'.join(legend_lines)
        
        # Add text box below plot with legend
        fig.text(0.05, 0.22, legend_text, fontsize=9, family='monospace',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9),
                verticalalignment='top')
        
        plt.show()

In [ ]:
import ast
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# create 29x2 subplot grid for all subjects (BR and Replay)
cmap = ListedColormap(['#1f77b4', '#1fa832', "#ce1818"])
atlas_labels = features.labels

# get all unique subjects (test with first 3)
all_plot_subjects = df_joint['subject'].unique()
n_subjects = len(all_plot_subjects)

# Create legend patches
legend_handles = [
    Patch(color='#1f77b4', label='(E) Encoding only'),
    Patch(color="#1fa832", label='(D) Decoding only'),
    Patch(color="#ce1818", label='(B) Both')
]

# create large figure with subplots: 2 columns (BR, Replay)
fig, axes = plt.subplots(n_subjects, 2, figsize=(18, 6*n_subjects))

for subj_idx, subject_id in enumerate(all_plot_subjects):
    subject_rows = df_joint[df_joint['subject'] == subject_id]
    
    for cycle_idx, cycle_pair in enumerate(['BR', 'Replay']):
        ax = axes[subj_idx, cycle_idx]
        
        cycle_rows = subject_rows[subject_rows['cycle_pair'] == cycle_pair]
        
        if len(cycle_rows) == 0:
            ax.axis('off')
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            continue
        
        enc_indices = set()
        dec_indices = set()
        
        for _, row in cycle_rows.iterrows():
            enc = row['enc_feature_idx']
            if isinstance(enc, str):
                enc = ast.literal_eval(enc)
            enc_indices.update(enc)
            
            dec = row['dec_feature_idx']
            if isinstance(dec, str):
                dec = ast.literal_eval(dec)
            dec_indices.update(dec)
        
        # mutually exclusive categories
        only_enc = enc_indices - dec_indices
        only_dec = dec_indices - enc_indices
        both = enc_indices & dec_indices
        
        # build region-coded volume
        region_labels = np.zeros(n_regions, dtype=int)
        for idx in only_enc:
            region_labels[idx] = 1
        for idx in only_dec:
            region_labels[idx] = 2
        for idx in both:
            region_labels[idx] = 3
        
        subject_map = np.zeros_like(atlas_data)
        for region_idx in range(n_regions):
            region_label = region_idx + 1
            mask = atlas_data == region_label
            subject_map[mask] = region_labels[region_idx]
        
        subject_img = nib.Nifti1Image(subject_map, atlas_img.affine, atlas_img.header)
        
        # plot on the appropriate axis
        paradigm = 'Stimulus' if cycle_pair == 'Replay' else 'Response'
        display = plotting.plot_glass_brain(
            subject_img,
            display_mode='lyrz',
            colorbar=False,
            cmap=cmap,
            vmin=0.5,
            vmax=3.5,
            title=f'{subject_id} {cycle_pair} ({paradigm}) | # significant regions E:{len(only_enc)} D:{len(only_dec)} B:{len(both)}',
            threshold=0.5,
            axes=ax
        )
        
        # Add legend to the right of Replay plots
        if cycle_idx == 1:
            ax.legend(handles=legend_handles, loc='upper left', 
                     bbox_to_anchor=(1.02, 0.5), fontsize=9)
plt.show()

In [ ]:
from pathlib import Path
from IPython.display import display, HTML
from nilearn import plotting
from matplotlib.colors import ListedColormap
import ast
import numpy as np
import nibabel as nib

# Interactive surface maps for s23 (BR + Replay)
subject_id = 's23'
out_dir = Path('/gpfs01/bartels/user/smuehlinghaus/causalcoding/Results/Causal_Interpretation/interactive_surfaces_s23')
out_dir.mkdir(parents=True, exist_ok=True)

subject_rows = df_joint[df_joint['subject'] == subject_id]

# Define custom discrete colormap: 0=black (background), 1=blue (encoding), 2=green (decoding), 3=red (both)
category_colors = ['#1f77b4', '#1fa832', '#ce1818']
discrete_cmap = ListedColormap(category_colors)

views = {}
for cycle_pair in ['BR', 'Replay']:
    cycle_rows = subject_rows[subject_rows['cycle_pair'] == cycle_pair]

    enc_indices = set()
    dec_indices = set()

    for _, row in cycle_rows.iterrows():
        enc = row['enc_feature_idx']
        if isinstance(enc, str):
            enc = ast.literal_eval(enc)
        enc_indices.update(enc)

        dec = row['dec_feature_idx']
        if isinstance(dec, str):
            dec = ast.literal_eval(dec)
        dec_indices.update(dec)

    # Mutually exclusive categories
    only_enc = enc_indices - dec_indices
    only_dec = dec_indices - enc_indices
    both = enc_indices & dec_indices

    print(f'{subject_id} {cycle_pair} - Only encoding: {len(only_enc)} | Only decoding: {len(only_dec)} | Both: {len(both)}')

    # 0 = background, 1 = encoding only, 2 = decoding only, 3 = both
    region_labels = np.zeros(n_regions, dtype=int)
    for idx in only_enc:
        if 0 <= idx < n_regions:
            region_labels[idx] = 1
    for idx in only_dec:
        if 0 <= idx < n_regions:
            region_labels[idx] = 2
    for idx in both:
        if 0 <= idx < n_regions:
            region_labels[idx] = 3

    subject_map = np.zeros_like(atlas_data)
    for region_idx in range(n_regions):
        subject_map[atlas_data == (region_idx + 1)] = region_labels[region_idx]

    subject_img = nib.Nifti1Image(subject_map, atlas_img.affine, atlas_img.header)

    # Paradigm label
    paradigm = 'Stimulus-based' if cycle_pair == 'Replay' else 'Response-based'

    # Create visualization with discrete colormap
    view = plotting.view_img_on_surf(
        subject_img,
        threshold=0.5,
        surf_mesh='fsaverage',
        title=f'{cycle_pair}: Blue=Encoding | Green=Decoding | Red=Both',
        cmap=discrete_cmap,
        vmin=0.5,
        vmax=3.5,
        symmetric_cmap=False,
        colorbar=False
    )

    html_path = out_dir / f'{subject_id}_{cycle_pair}_interactive_surface.html'
    view.save_as_html(str(html_path))
    views[cycle_pair] = view

# Open interactive HTML views directly in notebook
for cycle_pair in ['BR', 'Replay']:
    display(HTML(f"<h3>{subject_id} {cycle_pair} interactive surface</h3>"))
    display(views[cycle_pair])

In [ ]:
# Additional slice-based interactive views with view_img
from nilearn import plotting

slice_views = {}
for cycle_pair in ['BR', 'Replay']:
    cycle_rows = subject_rows[subject_rows['cycle_pair'] == cycle_pair]

    enc_indices = set()
    dec_indices = set()

    for _, row in cycle_rows.iterrows():
        enc = row['enc_feature_idx']
        if isinstance(enc, str):
            enc = ast.literal_eval(enc)
        enc_indices.update(enc)

        dec = row['dec_feature_idx']
        if isinstance(dec, str):
            dec = ast.literal_eval(dec)
        dec_indices.update(dec)

    # Mutually exclusive categories
    only_enc = enc_indices - dec_indices
    only_dec = dec_indices - enc_indices
    both = enc_indices & dec_indices

    # Build region labels (same as before)
    region_labels = np.zeros(n_regions, dtype=int)
    for idx in only_enc:
        if 0 <= idx < n_regions:
            region_labels[idx] = 1
    for idx in only_dec:
        if 0 <= idx < n_regions:
            region_labels[idx] = 2
    for idx in both:
        if 0 <= idx < n_regions:
            region_labels[idx] = 3

    subject_map = np.zeros_like(atlas_data)
    for region_idx in range(n_regions):
        subject_map[atlas_data == (region_idx + 1)] = region_labels[region_idx]

    subject_img = nib.Nifti1Image(subject_map, atlas_img.affine, atlas_img.header)

    # Paradigm label
    paradigm = 'S' if cycle_pair == 'Replay' else 'R'

    # Create slice-based visualization with view_img
    slice_view = plotting.view_img(
        subject_img,
        threshold=0.5,
        title=f'{paradigm}: B=Enc, G=Dec, R=Both',
        cmap=discrete_cmap,
        vmin=0.5,
        vmax=3.5,
        symmetric_cmap=False,
        black_bg=True,
        colorbar=False,
    )

    html_path_slice = out_dir / f'{subject_id}_{cycle_pair}_interactive_slices.html'
    slice_view.save_as_html(str(html_path_slice))
    slice_views[cycle_pair] = slice_view

    print(f'Saved slice view: {html_path_slice}')

# Display interactive slice views
for cycle_pair in ['BR', 'Replay']:
    display(slice_views[cycle_pair])